## Set Parameters for notebook

In [7]:
p_tgt_db_name = 'SilverData'
p_src_db_name = 'BronzeData'
p_src_entity_name = ''
p_tgt_entity_name = ''
p_src_modified_field = ''

StatementMeta(, 1d23a40f-c63c-4f4f-827a-035566fe14d2, 10, Finished, Available, Finished)

In [8]:
from pyspark.sql.functions import concat, concat_ws, sha2, col, row_number, lit, when
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, TimestampType
import json
from delta.tables import *

StatementMeta(, 1d23a40f-c63c-4f4f-827a-035566fe14d2, 11, Finished, Available, Finished)

## Merge data from bronze to silver

In [9]:
# Set the SQL configuration to handle datetime rebasing
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# Set parameters
stg_table_name = f"{p_src_db_name}.{p_src_entity_name}"
tgt_table_name = f"{p_tgt_db_name}.{p_tgt_entity_name}"

# Check if bronze table exists, get data and add any processing columns
if spark.catalog.tableExists(stg_table_name):
    excluded_columns = {p_src_modified_field, "ELT_PIPELINE_RUN_ID"}
    filtered_columns = [f"`{col}`" for col in [col.name for col in spark.table(f"{p_src_db_name}.{p_src_entity_name}").schema] if col not in excluded_columns]
    concat_expression = "CONCAT_WS('|', " + ", ".join(filtered_columns) + ")"
    concat_expression_scd = "CONCAT_WS('|', " + ", ".join(filtered_columns + [f"`{p_src_modified_field}`"]) + ")" 
    b_query = f"""
        SELECT *, 
               CAST(SHA2({concat_expression}, 256) AS STRING) AS ELT_UNIQUE_KEY,
               CAST(SHA2({concat_expression_scd}, 256) AS STRING) AS ELT_SCD_HASH
        FROM {p_src_db_name}.{p_src_entity_name}
        """

    b_df = spark.sql(b_query)
    #display(b_df)

    # Check if we have created silver table to compare, else insert only first "batch" of bronze data
    if spark.catalog.tableExists(tgt_table_name):
        s_query = f"""
        SELECT *
        FROM {p_tgt_db_name}.{p_tgt_entity_name}
        where ELT_VALID_TO is NULL
        """
        s_df = spark.sql(s_query)

        # create temp staging df from bronze_df only selecting the next "batch" of changes to apply
        s_df_alias = s_df.select([col(col_name).alias(f"SILVER_DF_{col_name}") for col_name in s_df.columns])
        tmp_stg_df = b_df.join(s_df_alias, b_df.ELT_UNIQUE_KEY == s_df_alias.SILVER_DF_ELT_UNIQUE_KEY,'left')
        tmp_stg_df = tmp_stg_df.filter((col(p_src_modified_field) > col("SILVER_DF_ELT_VALID_FROM")) | col("SILVER_DF_ELT_VALID_FROM").isNull())
        tmp_stg_df = tmp_stg_df.select(b_df.columns)
        window_spec = Window.partitionBy("ELT_UNIQUE_KEY").orderBy(col(p_src_modified_field).asc())
        tmp_stg_df = tmp_stg_df.withColumn("ELT_ROW_NBR", row_number().over(window_spec)).filter(col("ELT_ROW_NBR") == 1)

        # get records that are "inserts" since they do not exist in the target or are newer "updates" to an existing record in target
        inserts_df = tmp_stg_df.join(s_df_alias, tmp_stg_df.ELT_UNIQUE_KEY == s_df_alias.SILVER_DF_ELT_UNIQUE_KEY, "left")
        inserts_df = inserts_df.filter((col("SILVER_DF_ELT_UNIQUE_KEY").isNull()) | 
                                    (col("SILVER_DF_ELT_UNIQUE_KEY").isNotNull() & (col("SILVER_DF_ELT_VALID_FROM") < col(p_src_modified_field))))
        inserts_df = inserts_df.select(tmp_stg_df.columns)
        inserts_df = inserts_df.withColumn("ELT_VALID_FROM",inserts_df[p_src_modified_field])
        inserts_df = inserts_df.withColumn("ELT_VALID_TO",lit(None).cast(TimestampType()))
        inserts_df = inserts_df.withColumn("Current_Record_Flag", when(col("ELT_VALID_TO").isNull(), 1).otherwise(0))

        # get records that are "updates" since they do exist in the target and should be marked as historical with an "ELT_VALID_TO" value
        updates_df = tmp_stg_df.join(s_df_alias, tmp_stg_df.ELT_UNIQUE_KEY == s_df_alias.SILVER_DF_ELT_UNIQUE_KEY, "inner")
        updates_df = updates_df.filter(col("SILVER_DF_ELT_VALID_FROM") < col(p_src_modified_field))
        updates_df.drop("ELT_SCD_HASH")
        updates_df = updates_df.withColumn("ELT_SCD_HASH",updates_df["SILVER_DF_ELT_SCD_HASH"])
        updates_df = updates_df.select(tmp_stg_df.columns)
        updates_df = updates_df.withColumn("ELT_VALID_FROM",updates_df[p_src_modified_field])
        updates_df = updates_df.withColumn("ELT_VALID_TO",updates_df[p_src_modified_field])
        updates_df = updates_df.withColumn("Current_Record_Flag", lit(0))
        merge_df = inserts_df.unionAll(updates_df)

        #Converting the Table into DeltaTable to perform Match, Merge & Update logic (It is only specific to Delta Tables).
        silver_delta = DeltaTable.forName(spark, tgt_table_name)
        target_columns = silver_delta.toDF().columns
        updated_dict_insert_clause = {col: f"src.`{col}`" if col in merge_df.columns else "NULL" for col in target_columns}
        silver_delta.alias('tgt') \
        .merge(
            merge_df.alias('src'),
            'tgt.ELT_SCD_HASH = src.ELT_SCD_HASH'
        ) \
        .whenMatchedUpdate(
            condition = "tgt.ELT_VALID_TO IS NULL",
            set = {
                "ELT_VALID_TO": "src.ELT_VALID_TO",
                "Current_Record_Flag": "src.Current_Record_Flag"
            }
        ) \
        .whenNotMatchedInsert(
            values = updated_dict_insert_clause
        ) \
        .execute()

    else:

        # use only the bronze_df data since this is the first time merging to silver and table doesn't yet exist
        window_spec = Window.partitionBy("ELT_UNIQUE_KEY").orderBy(col(p_src_modified_field).asc())
        tmp_stg_df = b_df.withColumn("ELT_ROW_NBR", row_number().over(window_spec)).filter(col("ELT_ROW_NBR") == 1)
        merge_df = tmp_stg_df.withColumn("ELT_VALID_FROM",tmp_stg_df[p_src_modified_field])
        merge_df = merge_df.withColumn("ELT_VALID_TO",lit(None).cast(TimestampType()))
        merge_df = merge_df.withColumn("Current_Record_Flag", when(col("ELT_VALID_TO").isNull(), 1).otherwise(0))

        # Dedup the merge_df
        dedup_df = merge_df.dropDuplicates()
        dedup_df.write.format("delta").option("mergeSchema","true").mode("overwrite").saveAsTable(tgt_table_name)

StatementMeta(, 1d23a40f-c63c-4f4f-827a-035566fe14d2, 12, Finished, Available, Finished)

## Collect ELT load time for futher use in the pipeline

In [10]:
from datetime import datetime

# Collect the Max load_datetime from the udpated target table
modified_field_value_df = spark.sql(f"SELECT date_format(max({p_src_modified_field}), 'yyyy-MM-dd HH:mm:ss') FROM {p_tgt_db_name}.{p_tgt_entity_name}")
elt_load_time_str = modified_field_value_df.collect()[0][0]

# Exit the notebook and pass the elt_load_time variable
mssparkutils.notebook.exit({"elt_load_time": elt_load_time_str})

StatementMeta(, 1d23a40f-c63c-4f4f-827a-035566fe14d2, 13, Finished, Available, Finished)

ExitValue: {'elt_load_time': '2025-05-05 14:20:59'}